# PrismLab P7 — Reproducible Research Synthesis v1.1

**Research closeout notebook · 2 November 2026 · FlowState / PrismLab**

This notebook synthesizes **six pinned Liquid AI model configurations** and the P1–P6 local CPU research. It separates:

- **Model implementation evidence** (configuration and source-level tracing),
- **Within-experiment observations** (P2 OCR; P3 speech; P4 audio/visual matched holdout),
- **Routing/reliability evidence** (P5 decision probes; P6 system integration), and
- **Presentation artifacts** (interactive SVG pages, Mermaid diagrams, README).

**Scientific boundaries:** The P2, P3 and P4 datasets, rubrics, timing denominators and tasks differ. Do **not** pool or rank all models across phases. Architecture overview arrows are not proof of exact tensor operations. The d1 confidence outputs are not established as production-calibrated probabilities. All inference was conducted locally; this notebook **does not start servers or download weights**.

**Primary local sources:** `docs/p2/README.md`, `docs/p3/README.md`, `docs/p4/P4_FINAL_REPORT.md`, `docs/p5/README.md`, `docs/p6/README.md`, `results/p7/architecture/`, `visuals/mermaid/p7_execution_v1/`. See the end for evidence gaps and release gates.

In [ ]:
from pathlib import Path
import os, json, csv, hashlib, collections
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Use from the PrismLab root OR notebooks/ directory, or set PRISMLAB_ROOT.
_candidates = []
if os.environ.get("PRISMLAB_ROOT"):
    _candidates.append(Path(os.environ["PRISMLAB_ROOT"]).expanduser())
_candidates.extend([Path.cwd(), *Path.cwd().parents])
ROOT = next((p.resolve() for p in _candidates if (p / "prismlab").is_dir() and (p / "results").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open inside lfm3-prismlab or set PRISMLAB_ROOT to that directory")
OUT_FIG = ROOT / "docs/p7/figures"
OUT_DATA = ROOT / "results/p7/final"
OUT_FIG.mkdir(parents=True, exist_ok=True)
OUT_DATA.mkdir(parents=True, exist_ok=True)
COLORS = {"mint":"#52efd0", "blue":"#67baff", "pink":"#d6a0ff", "ink":"#070d19", "muted":"#4b5c78"}
plt.rcParams.update({"figure.dpi":135,"savefig.dpi":180,"font.size":10,"axes.spines.top":False,"axes.spines.right":False})

def save_figure(name):
    plt.tight_layout()
    for ext in ("png", "svg"):
        plt.savefig(OUT_FIG / f"{name}.{ext}", bbox_inches="tight")
    plt.show()
    print("Saved:", name, "(PNG + SVG)")

print("PrismLab root:", ROOT)
print("Figure output:", OUT_FIG)

## 1. Artifact and provenance audit

Read required local inputs, record SHA-256 hashes and explicitly flag missing evidence. Existing P7 files remain untouched. Note that missing local-only recordings are not automatically research failures; only cited evidence should be used in public statements.

In [ ]:
model_keys = ["text", "vision", "audio", "omni", "d1", "d1_omni"]
phase_reports = {
    "P1": "docs/p1/README.md",
    "P2": "docs/p2/README.md",
    "P3": "docs/p3/README.md",
    "P4": "docs/p4/P4_FINAL_REPORT.md",
    "P5": "docs/p5/README.md",
    "P6": "docs/p6/README.md",
}
tracked = list(phase_reports.values()) + [
    "results/p7/architecture/component_maps.json",
    "results/p7/architecture/p7_2_source_trace.json",
    "docs/p7/P7_2_WIRING_TRACE.md",
    "visuals/PrismLabs_Architecture_Extended_P7_NAV_v2.html",
]
tracked += [f"results/p7/architecture/{k}_config.json" for k in model_keys]
tracked += [f"visuals/mermaid/p7_execution_v1/p7_{k}.mmd" for k in model_keys]
tracked += [f"visuals/p7/models/{k}_v2.html" for k in model_keys]

manifest = []
for relative in tracked:
    f = ROOT / relative
    manifest.append({
        "path":relative, "exists":f.is_file(),
        "sha256":hashlib.sha256(f.read_bytes()).hexdigest() if f.is_file() else None,
        "size_bytes":f.stat().st_size if f.is_file() else None,
    })
missing = [r["path"] for r in manifest if not r["exists"]]
print(f"Local source files present: {len(manifest)-len(missing)}/{len(manifest)}")
for item in missing: print("MISSING:", item)
(OUT_DATA / "provenance_manifest.json").write_text(json.dumps(manifest,indent=2)+"\n")
assert (ROOT / "results/p7/architecture/component_maps.json").exists(), "Missing P7 component maps"
assert (ROOT / "results/p7/architecture/p7_2_source_trace.json").exists(), "Missing P7 source trace"

## 2. Verify six reference revisions and components

These counts are **configuration declarations**—not measured CPU throughput, memory consumption or a verified forward dataflow. The configurations refer to the official model repos; they are not checksums of locally quantized GGUF weights.

In [ ]:
configs = json.loads((ROOT / "results/p7/architecture/component_maps.json").read_text())
source_trace = json.loads((ROOT / "results/p7/architecture/p7_2_source_trace.json").read_text())
assert set(configs) == set(model_keys) == set(source_trace)
expected_parts = {
    "text": [24], "vision": [14,24], "audio": [17,24,3],
    "omni": [14,17,24,3], "d1": [14,24], "d1_omni": [14,17,24,3],
}
for key in model_keys:
    actual = [int(p["layers"]) for p in configs[key]["parts"]]
    assert actual == expected_parts[key], (key, actual)
    assert source_trace[key]["config_matches_saved"] is True, key
    print(f"{key:8s} | rev {source_trace[key]['revision'][:12]} | ",
          ", ".join(f"{p['title']} {p['layers']}L" for p in configs[key]["parts"]))

## 3. Figure 1 — Component composition matrix

Rows represent model variants, columns show separately configured component groups. **Blank means component absent from this configuration**, not that the model cannot process the relevant task.

In [ ]:
groups = ["Text backbone","Vision encoder","Audio encoder","Depthformer"]
labels = [configs[k]["name"] for k in model_keys]
matrix=[]
for key in model_keys:
    by_title={p["title"]:int(p["layers"]) for p in configs[key]["parts"]}
    matrix.append([by_title.get(g,0) for g in groups])

fig, ax = plt.subplots(figsize=(9.8,4.6))
from matplotlib.colors import ListedColormap, BoundaryNorm
# Discrete cells highlight category presence; printed numbers give actual layer counts.
img=ax.imshow([[int(x>0) for x in row] for row in matrix],
              cmap=ListedColormap(["#e9eef8","#c9f5ec"]), vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(groups)), groups)
ax.set_yticks(range(len(labels)),labels)
for i,row in enumerate(matrix):
    for j,num in enumerate(row):
        ax.text(j,i,str(num) if num else "—",ha="center",va="center",weight="bold",color="#15324a")
ax.set_title("Configured layers by model family (official pinned configs)",loc="left",pad=14,weight="bold")
ax.tick_params(length=0)
save_figure("p7_01_model_component_matrix")

## 4. Figure 2 — LFM3-1.7B decoder operator schedule

Verify the 24-layer sequence from the saved official `text_config.json`. The decoder also applies pre-operator RMSNorm, a post-operator RMSNorm, two residual additions, an MLP and post-FFN RMSNorm when `use_sandwich_norm` is true. This figure shows the **configured operator schedule only**, not the internal circuit.

In [ ]:
cfg=json.loads((ROOT / "results/p7/architecture/text_config.json").read_text())
layers=cfg["layer_types"]
counts=collections.Counter(layers)
assert len(layers)==24
assert counts=={"conv":13,"sliding_attention":5,"full_attention":6}
assert cfg["use_sandwich_norm"] is True
color_by_type={"conv":COLORS["mint"],"sliding_attention":COLORS["pink"],"full_attention":COLORS["blue"]}
name_by_type={"conv":"Convolution","sliding_attention":"Sliding attention","full_attention":"Full attention"}
fig, ax=plt.subplots(figsize=(13,2.6))
for i,operator in enumerate(layers,1):
    ax.bar(i,1,width=.78,color=color_by_type[operator],edgecolor="#1b3151")
    ax.text(i,.52,str(i).zfill(2),ha="center",va="center",fontsize=8,weight="bold",color="#081426")
ax.set_xlim(.3,24.7);ax.set_ylim(0,1.2)
ax.set_yticks([]);ax.set_xticks([])
ax.legend(handles=[Patch(facecolor=color_by_type[k],label=f"{name_by_type[k]} ({counts[k]})") for k in color_by_type],ncol=3,loc="upper center",bbox_to_anchor=(.5,-.03),frameon=False)
ax.set_title("LFM3-1.7B · 24-layer hybrid schedule",weight="bold",loc="left",pad=12)
save_figure("p7_02_text_operator_schedule")

## 5. Canonical phase-specific benchmark comparison

Figures below reproduce **reported, phase-local aggregate values** from the existing committed research reports. They do not recompute raw accuracy, nor assume that independent phases are comparable. If the local reports change, update this explicit registry only after reconciling their underlying records. Reported medians have different timed sample counts. The P4 paired holdout has an **exact McNemar p-value of 0.25**, which does not demonstrate a statistically significant architecture advantage.

In [ ]:
# These are transcriptions from published PrismLab phase reports.
# Each comparison is VALID ONLY INSIDE its named phase.
benchmarks = [
    {"phase":"P2", "label":"Direct VL", "success":10,"total":10,"median_s":38.53,"timed_n":None,"source":"docs/p2/README.md","task":"10 synthetic image QA/OCR cases"},
    {"phase":"P2", "label":"VL OCR → Text", "success":9,"total":10,"median_s":65.72,"timed_n":None,"source":"docs/p2/README.md","task":"10 synthetic image QA/OCR cases"},
    {"phase":"P3", "label":"Native S2S", "success":24,"total":25,"median_s":13.95,"timed_n":23,"source":"docs/p3/README.md","task":"25 English audio QA/voice tasks"},
    {"phase":"P3", "label":"Modular V2.1", "success":22,"total":25,"median_s":20.18,"timed_n":24,"source":"docs/p3/README.md","task":"25 English audio QA/voice tasks"},
    {"phase":"P4", "label":"Native Omni", "success":25,"total":40,"median_s":20.919,"timed_n":None,"source":"docs/p4/P4_FINAL_REPORT.md","task":"40 paired spatial/AV holdout cases"},
    {"phase":"P4", "label":"Modular ASR → VL", "success":28,"total":40,"median_s":21.288,"timed_n":None,"source":"docs/p4/P4_FINAL_REPORT.md","task":"40 paired spatial/AV holdout cases"},
]
for rec in benchmarks:
    assert (ROOT / rec["source"]).is_file(), f"Missing source: {rec['source']}"
    assert 0 <= rec["success"] <= rec["total"]
    print(rec["phase"],rec["label"],f"{rec['success']}/{rec['total']}",
          f"{100*rec['success']/rec['total']:.1f}%",f"median={rec['median_s']:.3f}s")
with (OUT_DATA / "benchmark_summary.csv").open("w",newline="") as fh:
    writer=csv.DictWriter(fh,fieldnames=benchmarks[0].keys())
    writer.writeheader(); writer.writerows(benchmarks)

### Figure 3 — Strict success **within each experimental phase**

The x-axis groups are **different benchmarks**. Differences between P2, P3 and P4 groups must not be interpreted as model rankings or deployment-ready accuracy.

In [ ]:
fig, ax=plt.subplots(figsize=(10.7,4.5))
xs=[0,1,3,4,6,7]
bar_colors=[COLORS["mint"],COLORS["blue"]]*3
for x,r,col in zip(xs,benchmarks,bar_colors):
    rate=100*r["success"]/r["total"]
    ax.bar(x,rate,.74,color=col,edgecolor="#172a42")
    ax.text(x,rate+1.3,f"{r['success']}/{r['total']}",ha="center",fontsize=9,weight="bold")
ax.set_ylim(0,112);ax.set_ylabel("Strict task success (%)")
ax.set_xticks(xs,[r["label"] for r in benchmarks],rotation=10,ha="right")
for x,phase,task in [(0.5,"P2","OCR"),(3.5,"P3","Speech"),(6.5,"P4","AV holdout")]:
    ax.text(x,106,f"{phase} · {task}",ha="center",fontsize=9,weight="bold",color="#253955")
ax.set_title("Within-phase strict success · Different tasks and evaluation rubrics",loc="left",weight="bold")
ax.grid(axis="y",alpha=.15)
save_figure("p7_03_phase_local_strict_success")

### Figure 4 — Reported median elapsed latency

**Medians are not pooled.** P3 timed sample counts are explicitly 23 (native) and 24 (modular); recovered cases without full timestamps were excluded from timed medians. The other phase reports give aggregate median values. Hardware, inference prompts and timing semantics may differ across phases.

In [ ]:
fig, ax=plt.subplots(figsize=(10.7,4.5))
for x,r,col in zip(xs,benchmarks,bar_colors):
    value=r["median_s"]
    ax.bar(x,value,.74,color=col,edgecolor="#172a42")
    label=f"{value:.2f}s"+(f" (n={r['timed_n']})" if r["timed_n"] else "")
    ax.text(x,value+.65,label,ha="center",fontsize=8,weight="bold")
ax.set_ylim(0,78);ax.set_ylabel("Reported median total latency (s)")
ax.set_xticks(xs,[r["label"] for r in benchmarks],rotation=10,ha="right")
for x,phase in [(0.5,"P2"),(3.5,"P3"),(6.5,"P4")]:
    ax.text(x,73,phase,ha="center",fontsize=10,weight="bold")
ax.set_title("Phase-local timing comparison · Do not rank across benchmarks",loc="left",weight="bold")
ax.grid(axis="y",alpha=.15)
save_figure("p7_04_phase_local_latency")

## 6. P5 decision research and P6 integration — separate evidence categories

- **P5** investigates routing, abstention, selective decisions, and confidence. Its final results require the **local P5 analysis/records**, because the published `docs/p5/README.md` still describes an early pilot; no production calibration is established. Do not reframe a decision confidence as probability of being correct.
- **P6** documents six model paths passing direct and orchestrated local inference, with a final 90/90 test suite. **Unit/integration test success is not task accuracy**. It does not imply all six individual HTTP endpoints underwent identical end-to-end evaluation.

This cell audits the evidence and refuses to silently fill unavailable P5 numerical fields.

In [ ]:
# P5 canonical benchmark outputs are saved by the experiments/p5 scripts.
# Do not assume results/p5 contains them; enumerate both locations.
p5_locations = [
    ROOT / 'experiments/p5/results',
    ROOT / 'results/p5',
    ROOT / 'docs/p5/figures',
    ROOT / 'notebooks/p5',
]
for folder in p5_locations:
    if folder.exists():
        files = [p for p in folder.rglob('*') if p.is_file()]
        print('P5 available:', folder.relative_to(ROOT), len(files), 'files')
        for file in files[:10]:
            print('  ', file.relative_to(ROOT))
    else:
        print('P5 not found:', folder.relative_to(ROOT))

p6_text = (ROOT / 'docs/p6/README.md').read_text()
print('P6 report mentions 90/90:', '90/90' in p6_text)
print('P6 report lists six direct paths:',
      'Six of six models passed direct CPU inference' in p6_text)


### P5 evidence extraction (version 1.1 correction)

P5 experiment scripts write `*_summary.json` to `experiments/p5/results/`, **not necessarily** to `results/p5/`. Read those summaries as recorded evidence; prefer a clearly identified latest run per suite and preserve source path and SHA-256. A missing summary is reported as missing, never replaced by remembered values or chart image interpretation.

P5 routing pilots, synthetic audio comparisons and integrated gated recommendations have **different denominators and evaluation policies**. They must not be pooled with P2–P4 task success, and `AUTO_RECOMMEND` is not authorization for external actions. The P5 threshold was selected post hoc and is not calibrated for production.


In [ ]:
# Read P5 evidence, without executing any model, benchmark or external request.
p5_patterns = {
    'P5.2 routing pilot': 'p5_2_*_summary.json',
    'P5.3 routing stress': 'p5_3_*_summary.json',
    'P5.6b audio vs transcript oracle': 'p5_6b_audio_intent_*_summary.json',
    'P5.7 gated integration': 'p5_7_integration_*_summary.json',
}
p5_data = {}
p5_dir = ROOT / 'experiments/p5/results'
for suite, pattern in p5_patterns.items():
    matches = sorted(p5_dir.glob(pattern)) if p5_dir.exists() else []
    if not matches:
        print('MISSING P5 evidence:', suite, '(', pattern, ')')
        continue
    path = matches[-1]   # Timestamped names in source scripts; report selection explicitly.
    doc = json.loads(path.read_text())
    assert isinstance(doc, dict), path
    p5_data[suite] = {'record':doc, 'path':str(path.relative_to(ROOT)),
                       'sha256':hashlib.sha256(path.read_bytes()).hexdigest(),
                       'matching_runs':len(matches)}
    print('P5 evidence:', suite, '=>', path.relative_to(ROOT),
          f'(among {len(matches)} matching files)')

# Retain a concise, reproducible evidence record with direct source pointers.
p5_metrics = {}
for suite, item in p5_data.items():
    doc = item['record']
    if suite.startswith('P5.2'):
        fields = ['cases','successful','errors','route_accuracy','route_brier',
                  'latency_p50_s','latency_p95_s','failed_route_ids']
    elif suite.startswith('P5.3'):
        fields = ['total_cases','successful_cases','errors','routing_accuracy',
                  'routing_brier','latency_p50_s','latency_p95_s','routing_failures']
    elif suite.startswith('P5.6b'):
        fields = ['requested','valid','errors','audio','text_oracle',
                  'fully_paired_cases','prediction_disagreements']
    else:
        fields = ['total_cases','successful_records','request_errors','model_requests',
                  'blocked_without_model_call','auto_recommendations',
                  'review_decisions','labeled_inferences','labeled_accuracy',
                  'review_required_auto_accepted','threshold',
                  'threshold_was_selected_post_hoc']
    p5_metrics[suite] = {
        'source_path':item['path'], 'source_sha256':item['sha256'],
        'matching_runs':item['matching_runs'],
        'observations':{k:doc[k] for k in fields if k in doc},
    }
    print(suite, '=>', p5_metrics[suite]['observations'])
(OUT_DATA / 'p5_evidence_summary.json').write_text(
    json.dumps(p5_metrics,indent=2,default=str)+'\n')

# Include selected P5 source summaries in the immutable artifact checksum manifest.
manifest = json.loads((OUT_DATA / 'provenance_manifest.json').read_text())
known = {m['path'] for m in manifest}
for item in p5_metrics.values():
    src = item['source_path']
    if src not in known:
        fp = ROOT / src
        manifest.append({'path':src,'exists':True,'sha256':item['source_sha256'],
                         'size_bytes':fp.stat().st_size})
(OUT_DATA / 'provenance_manifest.json').write_text(
    json.dumps(manifest,indent=2)+'\n')
print('Saved P5 evidence summary; source files were not changed.')


### Figure 5 — P5 decision evidence (only when source summaries exist)

If the local P5.2, P5.3, and P5.6b JSON summaries are present, display the routing pilot and stress results as **different sets**, and the paired synthesized audio vs exact-text-oracle results separately. Chart height is descriptive, not a calibrated confidence score; do not present cross-suite results as a model ranking. If fewer than two datasets are available, the code will not create a misleading figure.


In [ ]:
# Chart separately sourced P5 decision results; omit missing suites rather than fill in guesses.
route_names, route_rates = [], []
for suite, key in [('P5.2 routing pilot','route_accuracy'),
                   ('P5.3 routing stress','routing_accuracy')]:
    item=p5_metrics.get(suite, {}).get('observations', {})
    rate=item.get(key)
    if isinstance(rate,(int,float)) and 0 <= rate <= 1:
        route_names.append(suite.split()[0]);route_rates.append(100*rate)

pair = p5_metrics.get('P5.6b audio vs transcript oracle',{}).get('observations',{})
audio_names,audio_rates=[],[]
for key,label in [('audio','Audio'),('text_oracle','Exact text oracle')]:
    rate=pair.get(key,{}).get('accuracy')
    if isinstance(rate,(int,float)) and 0 <= rate <= 1:
        audio_names.append(label);audio_rates.append(100*rate)

if route_rates or audio_rates:
    fig, axes=plt.subplots(1,2,figsize=(11.2,4.1))
    for ax,names,rates,title,col in [
        (axes[0],route_names,route_rates,'Routing: different test sets',COLORS['blue']),
        (axes[1],audio_names,audio_rates,'Synthetic audio vs text oracle',COLORS['mint'])]:
        if rates:
            bars=ax.bar(names,rates,width=.65,color=col,edgecolor='#152c47')
            for bar,val in zip(bars,rates):
                ax.text(bar.get_x()+bar.get_width()/2,val+1,f'{val:.1f}%',ha='center',fontsize=10)
        else:
            ax.text(.5,.5,'No verified summary available',transform=ax.transAxes,
                    ha='center',va='center',color=COLORS['muted'])
        ax.set_ylim(0,112);ax.set_ylabel('Reported accuracy (%)')
        ax.set_title(title,fontsize=12,fontweight='bold')
        ax.grid(axis='y',alpha=.1)
    fig.suptitle('P5 recorded exploratory decision results — non-comparable suites',
                 fontsize=13,fontweight='bold')
    save_figure('p7_05_p5_decision_evidence')
else:
    print('Figure 5 skipped: no recorded P5 summary files; do not invent values.')


## 7. Failure taxonomy and scientific limitations

Maintain distinct layers of failure: input/perception errors, reasoning or pipeline hand-off failures, output-format failures, safety-guard stops, unsupported multimodal temporal alignment, and decision confidence/calibration. These examples are **phase-reported**, not an exhaustive error dataset.

In [ ]:
failure_examples=[
    ("P2","Pipeline cascading / token budget","04_lab_schedule.png failed downstream text generation despite visual phrase recall","docs/p2/README.md"),
    ("P3","Instruction formatting","A correct semantic response can fail a solitary-token exact-output instruction","docs/p3/README.md"),
    ("P3","Guarded abort","Modular V2.1 G02 was intentionally halted under word-length safeguard","docs/p3/README.md"),
    ("P4","Unsupported spatial relation","Both approaches answered unanswerable spatial prompts too often; both 4/16 on that subset","docs/p4/P4_FINAL_REPORT.md"),
    ("P4","Temporal alignment","Four-case video pilot scored temporal ordering 2/4; AV timestamp handling not independently established","docs/p4/P4_FINAL_REPORT.md"),
    ("P5","Calibration / abstention","Thresholds are exploratory, not production calibrated; do not authorize external actions", "docs/p5/README.md"),
    ("P6","Integration / evidence","ASR error is not ruled out solely by a downstream evidence gate", "docs/p6/README.md"),
]
for phase,kind,example,source in failure_examples:
    print(f"{phase} | {kind}: {example} [{source}]")
with (OUT_DATA / "failure_taxonomy.csv").open("w",newline="") as fh:
    w=csv.writer(fh);w.writerow(["phase","category","example","source"]);w.writerows(failure_examples)

## 8. Release summary for README integration (generated, non-destructive)

This exports a **draft source-indexed summary**. Review it manually before changing the root README; the published root README currently contains outdated planned-model status text. Avoid claiming production readiness, full online hosting or benchmark superiority.

In [ ]:
parts=[
"# PrismLab P7 — Research Synthesis (generated draft)",
"", "## Scope and provenance",
"- Local, quantized CPU-first multimodal research workbench; no online model inference required for this analysis.",
"- Six model configs and reference-source revisions are pinned in `results/p7/architecture/`.",
"- The individual architecture pages are conceptual source-grounded overviews; exact tensor-graph auditing is not complete.",
"", "## Models and architecture",
]
for k in model_keys:
    cfg=configs[k]; revision=source_trace[k]["revision"]
    component_summary=", ".join(f"{p['title']}: {p['layers']} layers" for p in cfg["parts"])
    parts.append(f"- **{cfg['name']}** — {component_summary}. Official reference revision `{revision}`.")
parts += ["", "## Within-phase empirical evidence"]
for phase in ("P2","P3","P4"):
    comparisons=[r for r in benchmarks if r["phase"]==phase]
    parts.append(f"- **{phase}** ({comparisons[0]['task']}): " + "; ".join(
        f"{r['label']} {r['success']}/{r['total']} strict, median {r['median_s']:.3f}s" for r in comparisons)
        + f". Source: `{comparisons[0]['source']}`.")
parts += [
"- **P4 paired holdout**: McNemar exact p=0.25; no statistically established advantage. Source: `docs/p4/P4_FINAL_REPORT.md`.",
"- **P5**: routing, calibration and abstention are exploratory; canonical output summaries are indexed in `results/p7/final/p5_evidence_summary.json` when local P5 source JSONs are available. Final production calibration remains absent.",
"- **P6**: six direct and six orchestrated local inference paths tested; 90/90 final regression tests. This does not establish 90/90 task accuracy. Source: `docs/p6/README.md`.",
"", "## Research figures",
"- `docs/p7/figures/p7_01_model_component_matrix.svg`",
"- `docs/p7/figures/p7_02_text_operator_schedule.svg`",
"- `docs/p7/figures/p7_03_phase_local_strict_success.svg`",
"- `docs/p7/figures/p7_04_phase_local_latency.svg`",
"- `docs/p7/figures/p7_05_p5_decision_evidence.svg` (only if P5 source data exist)",
"", "## Limitations and next steps",
"- Synthetic/pilot samples and CPU timings are not representative production benchmarks.",
"- Different phase data and rubrics cannot be pooled into an overall model ranking.",
"- Decision confidence is not production calibrated; external actions require review.",
"- The root README needs the six-model deployment status corrected using P6 evidence.",
"- P4-E exploratory 18-case notes and final published P4 report must be reconciled before making claims about an 18-case evaluation.",
"- PG-Lite follows P7 closure: replay-first sandbox, then optional local live inference.",
]
summary_path=ROOT / "docs/p7/P7_RELEASE_SUMMARY.generated.md"
summary_path.write_text("\n".join(parts)+"\n")
print("Generated draft:",summary_path)


## 9. Final research quality gate

Run the offline regression suites separately from the repository root, inspect figure readability and the browser architecture diagrams, then commit **selectively** (the workspace previously contained untracked audio/video research assets). Never use `git add -A` in this project without reviewing the full working tree.

```bash
python3 -m unittest discover -s tests -p 'test_p7*.py' -v
python3 -m unittest discover -s tests -p 'test_p6*.py' -q
git diff --check
git status --short
```

**README fixes to complete manually:** (1) replace the stale planned/undeployed cells for Omni and the two d1 models with locally tested statuses; (2) link P4–P7 reports and P7 architecture HTML; (3) link the four generated research figures; (4) state limitations; (5) revise the P5 README from the locally verified P5 final analysis. **Do not auto-overwrite original reports.**

**Known documentation reconciliation:** P4 final report says its proposed 18 video validation clips were not evaluated, whereas the local P4-E notes may contain partial normalization/analysis. These belong in separately labeled exploratory results until manifests and actual evaluation records are reconciled.

**Playground handoff:** design PG-Lite as a deterministic `P6 router + replay records + guarded local live inference` shell. Keep animations in the frontend; never treat SVG animation as inference progress or a measurement unless it is tied to a real API event. Details should be specified separately before Codex writes the application.